# Wikibase Bot Account Test

**Project:** Tests  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6, 2026-07-29)

**Purpose:** Verify that your Wikibase bot credentials are valid and the account can authenticate and read data.  
Edit **cell 2** to point at a different Wikibase instance, then run all cells.

**No extra packages required** — uses Python stdlib only.

**Tests:**
- Login — authenticate via bot password credentials (`Username@BotName`)
- Read item Q1 — fetch the first item via the Wikibase API

---

## Prerequisites

### 1. Create a bot password

1. Log in to the Wikibase with your regular account.
2. Go to **Special:BotPasswords** (e.g. `https://wikibase.kewl.org/wiki/Special:BotPasswords`).
3. Enter a bot name (e.g. `TestBot`) and click **Create**.
4. Grant at minimum: **Edit existing pages** · **Create, edit, and move pages**.
5. Copy the credentials shown: `Username@BotName` and the generated password.

### 2. Add credentials

Either edit the values directly in **cell 2**, or add them to `.env` in the repository root:

```
WB_URL=https://wikibase.kewl.org
WB_USER=YourUsername@YourBotName
WB_PASSWORD=your-generated-bot-password
```

> **Never commit `.env` to version control.** It is listed in `.gitignore`.

In [ ]:
# ── Target Wikibase instance ──────────────────────────────────────────────────
# Edit these values to point at a different wiki.
# They can also be set in ../.env  (WB_URL, WB_USERNAME, WB_PASSWORD)

WB_URL      = 'https://wikibase.kewl.org'  # Wikibase base URL
WB_USER     = ''                           # Username@BotName from Special:BotPasswords
WB_PASSWORD = ''                           # generated bot password

# ── Load overrides from .env (stdlib only — no python-dotenv needed) ──────────
import os
from pathlib import Path

_env_file = Path('../.env')
if _env_file.exists():
    with open(_env_file) as _f:
        for _line in _f:
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _, _v = _line.partition('=')
                os.environ.setdefault(_k.strip(), _v.strip().strip('"\''))

WB_URL      = os.getenv('WB_URL',      WB_URL).rstrip('/')   # strip trailing slash
WB_USER     = os.getenv('WB_USERNAME', os.getenv('WB_USER', WB_USER))   # .env uses WB_USERNAME
WB_PASSWORD = os.getenv('WB_PASSWORD', WB_PASSWORD)
API         = f'{WB_URL}/w/api.php'

if not WB_USER or not WB_PASSWORD:
    raise EnvironmentError('Set WB_USERNAME and WB_PASSWORD in ../.env or directly in this cell.')

print(f'Target  : {WB_URL}')
print(f'API     : {API}')
print(f'Bot user: {WB_USER}')
print(f'Password: {"*" * len(WB_PASSWORD)}')

In [ ]:
import json, urllib.request, urllib.parse, http.cookiejar

# ── HTTP session (persists cookies across requests) ───────────────────────────
_jar    = http.cookiejar.CookieJar()
_opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(_jar))

def _get(params):
    url = API + '?' + urllib.parse.urlencode({**params, 'format': 'json'})
    with _opener.open(url, timeout=15) as r:
        return json.loads(r.read().decode())

def _post(data):
    body = urllib.parse.urlencode({**data, 'format': 'json'}).encode()
    with _opener.open(urllib.request.Request(API, data=body), timeout=15) as r:
        return json.loads(r.read().decode())

# ── Test runner ───────────────────────────────────────────────────────────────
_results = []
_state   = {}

def _test(label, fn):
    try:
        detail = fn()
        _results.append((label, True))
        print(f'  PASS  {label}' + (f'  ->  {detail}' if detail else ''))
    except Exception as exc:
        _results.append((label, False))
        print(f'  FAIL  {label}  ->  {exc}')

# ── Tests ─────────────────────────────────────────────────────────────────────
_results.clear()
_state.clear()
print(f'Bot account test: {WB_USER}')
print(f'Target          : {WB_URL}')
print('─' * 60)

# Test 1 — Login
def t1():
    login_token = _get({'action': 'query', 'meta': 'tokens', 'type': 'login'})['query']['tokens']['logintoken']
    r = _post({'action': 'login', 'lgname': WB_USER, 'lgpassword': WB_PASSWORD, 'lgtoken': login_token})
    if r.get('login', {}).get('result') != 'Success':
        raise ValueError(r.get('login'))
    _state['csrf'] = _get({'action': 'query', 'meta': 'tokens', 'type': 'csrf'})['query']['tokens']['csrftoken']
    return f'logged in as {r["login"]["lgusername"]}'
_test('Login', t1)

# Test 2 — Read item Q1
def t2():
    r = _get({'action': 'wbgetentities', 'ids': 'Q1', 'props': 'labels'})
    if 'error' in r:
        raise ValueError(r['error']['info'])
    entity = r['entities'].get('Q1', {})
    if 'missing' in entity:
        return 'Q1 not found (triplestore may be empty)'
    labels = {lang: v['value'] for lang, v in entity.get('labels', {}).items()}
    return f'Q1 labels: {labels or "(none)"}'
_test('Read item Q1', t2)

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(1 for _, ok in _results if ok)
total  = len(_results)
print('─' * 60)
verdict = 'ALL PASS ✓' if passed == total else f'FAILED {total - passed}/{total} ✗'
print(f'Result: {passed}/{total} passed  |  {verdict}')